In [37]:
from pathlib import Path
import pandas as pd
import sys


In [38]:
FOLDER_CITESEQ = "../results/citeseq/k_ablation/25d/"
FOLDER_NEOCORTEX = "../results/neo_cortex/k_ablation/15d/"

In [39]:
# Execute the R clustering script for every benchmark subfolder that contains a denoising .pt file
from pathlib import Path
import subprocess, os

# relative to this notebook (benchmarks/), adjust if you run notebook from a different cwd
bench_root = Path('../benchmarks')
if not bench_root.exists():
    raise FileNotFoundError(f'Benchmarks folder not found: {bench_root.resolve()}')

# patterns to detect denoising outputs (flexible)
rscript = Path('..') / 'scripts' / 'clustering_metrics.R'
if not rscript.exists():
    print(f'Warning: R script not found at {rscript}. Adjust the path if necessary.')

# Iterate immediate subfolders of benchmarks/ (this notebook lives in benchmarks/)
for sub in [*bench_root.glob("**/denoising_results.pt"), Path(FOLDER_CITESEQ + "denoising_results.pt"), Path(FOLDER_NEOCORTEX + "denoising_results.pt")]:
    print('\n--- Running R script for folder:', sub)
    cmd = ['Rscript', str(rscript), str(sub.parent.resolve())]
    env = os.environ.copy()
    env['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

    try:
        res = subprocess.run(cmd, env=env, capture_output=True, text=True, check=False)
        print('Return code:', res.returncode)
        if res.stdout:
            print('--- R stdout ---')
            print(res.stdout)
        if res.stderr:
            print('--- R stderr ---')
            print(res.stderr)
    except FileNotFoundError as e:
        print('Failed to run Rscript (not found):', e)
    except Exception as e:
        print('Error running Rscript for', sub, e)


--- Running R script for folder: ../benchmarks/citeseq/pca/denoising_results.pt
Return code: 0
--- R stdout ---
Using folder: /Users/dm954/Documents/code/mixed_diffusion/benchmarks/citeseq/pca 
Denoised data shape: 1000 25 
True data shape: 1000 25 
Extracted true labels from denoising_results.pt
Class of cell_names: character 
Length of cell_names: 1000 
Class of unique_labels: character 
Unique labels: B naive, CD14 Mono, CD4 TCM, CD4 TEM, NK, gdT, CD16 Mono, CD4 Naive, Plasmablast, B memory, Platelet, CD8 Naive, CD8 TCM, CD8 TEM, Treg, Eryth, ILC, pDC, B intermediate, MAIT, HSPC, CD8 Proliferating, CD4 CTL, dnT, cDC2, cDC1, NK Proliferating, NK_CD56bright, CD4 Proliferating 
Label distribution:
cell_names
   B intermediate          B memory           B naive         CD14 Mono 
               15                21                49               271 
        CD16 Mono           CD4 CTL         CD4 Naive CD4 Proliferating 
               40                11               113         

In [40]:
# Run gather benchmarks script for citeseq and neo_cortex

def aggregate_benchmarks(benchmarks_root: Path):
    benchmarks_root = Path(benchmarks_root)
    print(benchmarks_root)
    if not benchmarks_root.exists():
        raise SystemExit(f"Benchmarks folder not found: {benchmarks_root}")

    frames = []
    for csv_path in benchmarks_root.glob("**/clustering_metrics_results.csv"):
        try:
            df = pd.read_csv(csv_path)
            parent_folder_name = csv_path.parent.name
            df["method"] = parent_folder_name
            df["benchmark_folder"] = str(csv_path)
            frames.append(df)
        except Exception as e:
            print(f"Failed to read {csv_path}: {e}")

    if not frames:
        print("No clustering_metrics_results.csv files found in any subfolder.")
        return

    return pd.concat(frames, ignore_index=True)

In [41]:
df_citeseq = aggregate_benchmarks("citeseq")

# add the dice runs
df_dice = pd.read_csv(FOLDER_CITESEQ + "clustering_metrics_results.csv")
df_dice["method"] = "Dice"
df_citeseq = pd.concat([df_citeseq, df_dice])

df_neocortex = aggregate_benchmarks("neo_cortex")
df_dice = pd.read_csv(FOLDER_NEOCORTEX + "clustering_metrics_results.csv")
df_dice["method"] = "Dice"
df_neocortex = pd.concat([df_neocortex, df_dice])

citeseq
neo_cortex


In [42]:
df_neocortex

,Dataset,Metric,Value,method,benchmark_folder
0,Denoised,Adjusted_Rand_Index,0.346589,pca,neo_cortex/pca/clustering_metrics_results.csv
1,Denoised,V_Measure,0.524726,pca,neo_cortex/pca/clustering_metrics_results.csv
2,Denoised,Normalized_Mutual_Information,0.495867,pca,neo_cortex/pca/clustering_metrics_results.csv
3,Denoised,Mean_LISI_Score,1.495947,pca,neo_cortex/pca/clustering_metrics_results.csv
4,Denoised,Mean_Silhouette_Score,0.428010,pca,neo_cortex/pca/clustering_metrics_results.csv
...,...,...,...,...,...
7,True,V_Measure,0.505937,Dice,NaN
8,True,Normalized_Mutual_Information,0.441756,Dice,NaN
9,True,Mean_LISI_Score,1.395275,Dice,NaN
10,True,Mean_Silhouette_Score,0.429087,Dice,NaN


In [43]:
df_neocortex = df_neocortex[df_neocortex["Dataset"] == "Denoised"]
neocortex_results = df_neocortex.pivot_table(values="Value", index="method",columns="Metric")

In [44]:
df_citeseq = df_citeseq[df_citeseq["Dataset"] == "Denoised"]
df_citeseq

,Dataset,Metric,Value,method,benchmark_folder
0,Denoised,Adjusted_Rand_Index,0.744862,pca,citeseq/pca/clustering_metrics_results.csv
1,Denoised,V_Measure,0.787320,pca,citeseq/pca/clustering_metrics_results.csv
2,Denoised,Normalized_Mutual_Information,0.689037,pca,citeseq/pca/clustering_metrics_results.csv
3,Denoised,Mean_LISI_Score,1.333594,pca,citeseq/pca/clustering_metrics_results.csv
4,Denoised,Mean_Silhouette_Score,0.475918,pca,citeseq/pca/clustering_metrics_results.csv
...,...,...,...,...,...
1,Denoised,V_Measure,0.813389,Dice,NaN
2,Denoised,Normalized_Mutual_Information,0.740225,Dice,NaN
3,Denoised,Mean_LISI_Score,1.343875,Dice,NaN
4,Denoised,Mean_Silhouette_Score,0.570955,Dice,NaN


In [45]:
citeseq_results = df_citeseq.pivot_table(values="Value", index="method",columns="Metric")

In [62]:
def df_to_latex_formatted(df, float_precision=3, bold_best=True, maximize=True, 
                          output_file=None):
    """
    Convert DataFrame to LaTeX with float precision and bolding best scores.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to convert (should have numeric columns)
    float_precision : int
        Number of decimal places for floats (default: 3)
    bold_best : bool
        Whether to bold the best score in each column (default: True)
    maximize : bool
        If True, bold the maximum value; if False, bold minimum (default: True)
    output_file : str or None
        Path to save LaTeX output. If None, just returns string
        
    Returns
    -------
    str
        LaTeX table string
    """
    
    # Make a copy to avoid modifying original
    df_copy = df.copy()
    
    # Find best values per column if bolding
    best_indices = {}
    if bold_best:
        for col in df_copy.columns:
            if pd.api.types.is_numeric_dtype(df_copy[col]):
                if maximize:
                    best_idx = df_copy[col].idxmax()
                else:
                    best_idx = df_copy[col].idxmin()
                best_indices[col] = best_idx
    
    # Format float values to specified precision
    for col in df_copy.columns:
        if pd.api.types.is_numeric_dtype(df_copy[col]):
            # Format as strings with specified precision
            df_copy[col] = df_copy[col].apply(
                lambda x: f"{x:.{float_precision}f}" if pd.notna(x) else ""
            )
    
    # Bold the best values
    if bold_best:
        for col, best_idx in best_indices.items():
            current_val = df_copy.loc[best_idx, col]
            df_copy.loc[best_idx, col] = f"\\textbf{{{current_val}}}"
    
    # Convert to LaTeX
    return df_copy.to_latex(escape=False, index=True)

columns = ['Adjusted_Rand_Index', 'Mean_LISI_Score', 'Normalized_Mutual_Information', 'V_Measure']



In [63]:
# Generate LaTeX for selected columns and write to files
citeseq_tex = df_to_latex_formatted(citeseq_results[columns])
neocortex_tex = df_to_latex_formatted(neocortex_results[columns])

citeseq_out = Path("clustering_metrics_table_citeseq.tex")
neocortex_out = Path("clustering_metrics_table_neocortex.tex")

citeseq_out.write_text(citeseq_tex)
neocortex_out.write_text(neocortex_tex)

print(f"Wrote LaTeX tables to:\n- {citeseq_out.resolve()}\n- {neocortex_out.resolve()}")

Wrote LaTeX tables to:
- /Users/dm954/Documents/code/mixed_diffusion/benchmarks/clustering_metrics_table_citeseq.tex
- /Users/dm954/Documents/code/mixed_diffusion/benchmarks/clustering_metrics_table_neocortex.tex
